# Ch.2 — From VAEs to DDPM

**Goal:** ground the ELBO machinery on a VAE, then build DDPM from scratch
twice (2D toy data → MNIST), reproducing the derivations in Ch. 2.1–2.2.

**Structure**
1. Part A — VAE on MNIST (ELBO, reparameterization trick)
2. Part B — DDPM theory recap (forward process, Lemma 2.2.2, ε/x₀ equivalence)
3. Part C — DDPM from scratch on 2D toy data (MLP, ancestral sampling)
4. Part D — DDPM on MNIST (small conv-U-Net)
5. Checkpoint — visual + quantitative sanity checks

In [ ]:
# --- Setup (PROVIDED): datasets, PyTorch, plotting -----------------------
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Complete TODOs sequentially. SELF-CHECK cells do not need full training;
# train VAE/DDPM only after each section passes. NotImplementedError is
# intentional: it marks a coding exercise, not a faulty notebook.
FAST_DEV_RUN = False
VAE_EPOCHS = 1 if FAST_DEV_RUN else 10
TOY_TRAIN_STEPS = 100 if FAST_DEV_RUN else 7000
MNIST_EPOCHS = 1 if FAST_DEV_RUN else 5
# MNIST training is computationally expensive; GPU is recommended.


## 2.1 Variational Autoencoder

A VAE assumes a latent-variable model $p(x, z) = p(x|z) \cdot p(z)$, with prior
$p(z) = N(0, I)$. Since the true posterior $p(z|x)$ is intractable, we
introduce an encoder $q_θ(z|x)$ and optimize the **Evidence Lower Bound**:

$$L_{ELBO}(x) = E_{q_θ(z|x)}[ log p_ϕ(x|z) ]  −  D_{KL}( q_θ(z|x) ‖ p(z) )$$

- First term: **reconstruction** (decoder likelihood).
- Second term: **regularizer** pulling the encoder's output toward the prior.

With $q_θ(z|x) = N(z; μ_θ(x), diag(σ_θ²(x)))$, the KL term is closed-form
(Gaussian-vs-standard-Gaussian KL), and sampling $z$ uses the
**reparameterization trick**: $z = μ + σ ⊙ ε, ε \sim N(0, I)$, which keeps the
sampling step differentiable w.r.t. θ.

This is the same "encoder maps data forward, decoder maps it back, bound
the log-likelihood" template that DDPM reuses in Part B — except DDPM fixes
the encoder and stacks many latent layers.

In [ ]:
# Data
transform = transforms.ToTensor()
train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
# OPTIONAL TODO: inspect x.shape, x.min(), x.max() for one minibatch.


In [ ]:
# --- TODO 1: Gaussian VAE, reparameterization and ELBO ------------------
class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        # TODO 1.1: Create a shared encoder feature stack:
        # Flatten -> Linear(784,256) -> ReLU -> Linear(256,128) -> ReLU.
        # Then create independent Linear(128,latent_dim) heads for
        # mu and logvar (LOG variance, not standard deviation).
        raise NotImplementedError("TODO 1.1: encoder + Gaussian heads")

        # TODO 1.2: Create decoder network with input `latent_dim`,
        # hidden sizes 128 and 256 (ReLU) and final 784 output units.
        # Apply Sigmoid at the end so pixel probabilities are in [0,1].
        raise NotImplementedError("TODO 1.2: decoder MLP")

    def encode(self, x):
        # TODO 1.3: Compute shared features, return (mu, logvar) [B,latent_dim].
        raise NotImplementedError("TODO 1.3: encode")

    def reparameterize(self, mu, logvar):
        # TODO 1.4: std=exp(0.5*logvar); eps=randn_like(std);
        # return mu+eps*std. This preserves gradient flow to both heads.
        raise NotImplementedError("TODO 1.4: differentiable latent sampling")

    def decode(self, z):
        # TODO 1.5: Decode z and reshape [B,784] to [B,1,28,28].
        raise NotImplementedError("TODO 1.5: pixel decoder")

    def forward(self, x):
        # TODO 1.6: Call encode -> reparameterize -> decode;
        # return (x_hat, mu, logvar) in the ORIGINAL order.
        raise NotImplementedError("TODO 1.6: VAE forward")


def vae_loss(x_hat, x, mu, logvar):
    # TODO 1.7: Implement negative ELBO:
    #   BCE reconstruction, reduction='sum' over all samples/pixels
    #   + KL(q(z|x)||N(0,I)) = -0.5 * sum(1+logvar-mu^2-exp(logvar)).
    # Divide EACH term by batch size and return (total,recon,kl).
    # HINT: flatten images to [B,784] for binary_cross_entropy.
    raise NotImplementedError("TODO 1.7: VAE reconstruction+KL loss")


In [ ]:
# --- SELF-CHECK 1: VAE shapes, BCE/KL, differentiable reparameterization -
net=VAE(latent_dim=8).to(device)
x=torch.rand(5,1,28,28,device=device)
x_hat,mu,logvar=net(x)
assert x_hat.shape==x.shape and mu.shape==(5,8) and logvar.shape==(5,8)
assert ((x_hat>=0)&(x_hat<=1)).all()
loss,recon,kl=vae_loss(x_hat,x,mu,logvar)
assert all(v.ndim==0 and torch.isfinite(v) for v in (loss,recon,kl))
assert torch.isclose(loss,recon+kl,rtol=1e-5,atol=1e-5)
assert kl>=0
loss.backward()
assert any(p.grad is not None and torch.isfinite(p.grad).all()
           for p in net.parameters())
zeros=torch.zeros(5,8,device=device)
klzero=vae_loss(x,x,zeros,zeros)[2]
assert klzero.abs()<1e-6
print("✓ SELF-CHECK 1: VAE I/O, ELBO, KL=0 for standard normal, gradients")


In [ ]:
# --- TODO 2: reusable VAE batch-training function -----------------------
def vae_train_step(model, optimizer, x):
    """Return (total_loss,reconstruction_loss,kl_loss) for this minibatch."""
    # TODO 2.1: Forward x through model, unpack (x_hat, mu, logvar).
    # TODO 2.2: Call vae_loss(x_hat,x,mu,logvar).
    # TODO 2.3: zero_grad -> loss.backward() -> optimizer.step().
    # TODO 2.4: Return the three scalar loss tensors (total,recon,kl).
    raise NotImplementedError("TODO 2.1–2.4: reusable VAE train step")


In [ ]:
# --- SELF-CHECK 2: one VAE update on synthetic images (no downloads) ----
model_test = VAE(latent_dim=8).to(device)
opt_test = torch.optim.Adam(model_test.parameters(), lr=1e-3)
x_test = torch.rand(8,1,28,28,device=device)
before = [p.detach().clone() for p in model_test.parameters()]
loss,recon,kl=vae_train_step(model_test,opt_test,x_test)
assert torch.isfinite(loss) and torch.isfinite(recon) and torch.isfinite(kl)
assert any(not torch.equal(old,p) for old,p in zip(before,model_test.parameters()))
print("✓ SELF-CHECK 2: VAE update changes weights")


In [ ]:
# --- Train VAE after SELF-CHECK 2 ----------------------------------------
vae = VAE().to(device)
# TODO 2.5: Initialize optimizer = Adam(vae.parameters(),lr=1e-3).
raise NotImplementedError("TODO 2.5: VAE optimizer")
for epoch in range(VAE_EPOCHS):
    total = 0.0
    for x,_ in train_loader:
        x=x.to(device)
        # TODO 2.6: Call your vae_train_step(vae,optimizer,x), and
        # accumulate batch loss weighted by x.size(0) into total.
        raise NotImplementedError("TODO 2.6: train VAE and aggregate loss")
    print(f"epoch {epoch} | ELBO loss {total/len(train_ds):.2f}")


In [ ]:
vae.eval()
with torch.no_grad():
    z = torch.randn(16, 16).to(device)
    samples = vae.decode(z).cpu()

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i, 0], cmap="gray"); ax.axis("off")
plt.suptitle("VAE prior samples")
plt.show()
# OPTIONAL TODO: compare two sets of generated images with distinct z seeds.


**Observation:** VAE samples are blurry — a single Gaussian decoder step has
to hallucinate all the detail at once. A **Hierarchical VAE** stacks latents
`z_1..z_L` to spread this over multiple stages, but training becomes unstable
(deep latents get weak gradient signal; shallow layers dominate reconstruction).

DDPM keeps the hierarchical, multi-step structure but makes one crucial
simplification: **fix the encoder** to a known, non-learned noising process,
and only learn the reverse (decoder) chain. That's Section 2.2.

## 2.2 DDPM

**Forward process (fixed encoder).** Gaussian transition kernel:

$$p(x_i \mid x_{i-1}) = \mathcal{N}\left(x_i ; \alpha_i x_{i-1}, \beta_i^2 I\right), \quad \text{with } \alpha_i = \sqrt{1-\beta_i^2}$$

Closing this recursively gives a direct jump from data to any noise level $i$:

$$x_i = \bar\alpha_i x_0 + \sqrt{1-\bar\alpha_i^2}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I), \qquad \bar\alpha_i = \prod_{k \le i} \alpha_k$$

This lets us sample any $x_i$ from $x_0$ in one shot — no need to simulate
the chain step by step during training.

**Reverse process (learnable decoder).** We want $p_\phi(x_{i-1} \mid x_i)$. The
*true* posterior conditioned on $x_0$ is tractable (Lemma 2.2.2):

$$p(x_{i-1} \mid x_i, x_0) = \mathcal{N}\left(x_{i-1}; \mu(x_i, x_0, i), \sigma^2(i) I\right)$$

so we parameterize $p_\phi(x_{i-1} \mid x_i) = \mathcal{N}\left(x_{i-1}; \mu_\phi(x_i,i), \sigma^2(i) I\right)$ and
match means via a KL loss — this is $L_{\text{DDPM}}$ in Eq. (2.2.8).

**$\epsilon$-prediction.** Substituting the closed-form $x_i$ back into $\mu$ gives a
noise-prediction parameterization: instead of predicting $\mu$ directly, a
network $\epsilon_\phi(x_i, i)$ predicts the noise that was added. The loss reduces to
plain MSE and is *equivalent up to a reweighting* to the original ELBO term:

$$L_{\text{simple}}(\phi) = \mathbb{E}_{i, x_0, \epsilon} \left[ \left\| \epsilon_\phi(x_i, i) - \epsilon \right\|^2 \right], \qquad x_i = \bar\alpha_i x_0 + \sqrt{1-\bar\alpha_i^2}\, \epsilon$$

This is what we implement below. ($x$-prediction and $v$-prediction are
algebraically equivalent reparameterizations — Week 6 revisits this.)

**Sampling (ancestral).** Starting from $x_L \sim \mathcal{N}(0,I)$, repeatedly:
1. Predict $\epsilon_\phi(x_i, i)$.
2. Estimate the mean of $p_\phi(x_{i-1} \mid x_i)$ from it.
3. Sample $x_{i-1}$ by adding calibrated noise $\sigma(i) z$, $z \sim \mathcal{N}(0,I)$ (except at $i=1$).


In [ ]:
# --- TODO 3: DDPM noise schedule and one-shot forward kernel ------------
def make_schedule(T=1000, beta_start=1e-4, beta_end=0.02):
    """Return (betas, alphas, alpha_bars), each shape [T]."""
    # TODO 3.1: Use torch.linspace for increasing betas (noise variance).
    # Compute per-step alphas = 1 - betas.
    # Compute cumulative alpha_bars = torch.cumprod(alphas,dim=0).
    # Important: this notebook uses alphas = 1-beta (VARIANCES), not
    # alpha=sqrt(1-beta) as the signal coefficient in the book's notation.
    raise NotImplementedError("TODO 3.1: beta/alpha/alpha_bar")

T = 1000
betas, alphas, alpha_bars = [t.to(device) for t in make_schedule(T)]


def q_sample(x0, t, noise=None):
    """Return (x_t, noise) for integer timesteps t, shape [B]."""
    # TODO 3.2: Use torch.randn_like(x0) only if noise is None.
    # Index alpha_bars[t] to obtain a length-B vector.
    # TODO 3.3: Reshape it to [B,1] for toy points or [B,1,1,1]
    # for images. HINT: x0.ndim controls number of singleton axes.
    # Compute x_t = sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*noise.
    # Return x_t and the ACTUAL noise used (not fresh noise).
    raise NotImplementedError("TODO 3.2–3.3: closed-form forward diffusion")


In [ ]:
# --- SELF-CHECK 3: alpha-bar algebra and 2D/4D broadcasting -------------
beta,alpha,ab=make_schedule(T=12, beta_start=1e-4,beta_end=0.02)
assert beta.shape==alpha.shape==ab.shape==(12,)
assert torch.all(beta[1:]>=beta[:-1]) and torch.all(ab[1:]<ab[:-1])
torch.testing.assert_close(ab,torch.cumprod(1-beta,dim=0))
for dims in [(4,2),(4,1,28,28)]:
    clean=torch.randn(*dims,device=device)
    eps=torch.ones_like(clean)
    ids=torch.tensor([0,10,11,5],device=device)
    noisy, returned=q_sample(clean,ids,noise=eps)
    broadcast=alpha_bars[ids].view(-1,*([1]*(clean.ndim-1)))
    expected=broadcast.sqrt()*clean+(1-broadcast).sqrt()*eps
    assert noisy.shape==clean.shape
    torch.testing.assert_close(noisy,expected)
    torch.testing.assert_close(returned,eps)
print("✓ SELF-CHECK 3: schedule identity + 2D/image forward noise")


### 2.2.1 DDPM on 2D toy data
Before touching MNIST, we train and sample DDPM on a synthetic 2D
distribution. Therefore, we can see the whole distribution at once so we can debug easily.

In [ ]:
# Toy dataset
X, _ = make_moons(n_samples=4000, noise=0.05)
X = torch.tensor(X, dtype=torch.float32)
X = (X - X.mean(0)) / X.std(0)  # normalize to roughly unit variance

plt.scatter(X[:, 0], X[:, 1], s=3); plt.title("Target: two moons"); plt.show()
# OPTIONAL TODO: compare an unnormalized vs normalized moons dataset.


In [ ]:
# --- TODO 4: sinusoidal timestep embedding and epsilon-predictor --------
class SinusoidalEmbed(nn.Module):
    """Encode integer t [B] as sinusoidal features [B,dim]."""
    def __init__(self, dim=32):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # TODO 4.1: half=dim//2. Create exponentially spaced frequencies
        # from ~1 down to ~1/10000 (see Ch. 2 embedding code).
        # phases [B,half] = t[:,None].float()*freqs[None].
        # Concatenate sin(phases), cos(phases) into [B,dim].
        raise NotImplementedError("TODO 4.1: sinusoidal timestep features")


class ToyEpsNet(nn.Module):
    """Noise prediction epsilon_phi(x_t,t): [B,2] x [B] -> [B,2]."""
    def __init__(self, dim=2, hidden=128, t_dim=32):
        super().__init__()
        # TODO 4.2: Create SinusoidalEmbed(t_dim), then an MLP with
        # input dim+t_dim, two hidden layers of width hidden (SiLU), and
        # final output of size dim (no output activation).
        raise NotImplementedError("TODO 4.2: conditional epsilon-predictor MLP")

    def forward(self, x, t):
        # TODO 4.3: Concatenate x with time embedding on last dimension,
        # pass result through the MLP. Output shape matches x.
        raise NotImplementedError("TODO 4.3: timestep conditioning")


In [ ]:
# --- SELF-CHECK 4: embed shape and epsilon-model gradients --------------
test_times=torch.tensor([0,50,500,999],device=device)
emb=SinusoidalEmbed(32).to(device)(test_times)
assert emb.shape==(4,32) and torch.isfinite(emb).all()
net=ToyEpsNet(hidden=16).to(device)
x=torch.randn(4,2,device=device)
y=net(x,test_times)
assert y.shape==x.shape and torch.isfinite(y).all()
y.square().mean().backward()
assert any(p.grad is not None for p in net.parameters())
print("✓ SELF-CHECK 4: embedding/epsilon-predictor shapes and gradients")


In [ ]:
# --- TODO 5: reusable epsilon-prediction training step ------------------
def ddpm_train_step(model, optimizer, x0):
    """Train on toy points OR images; return scalar noise-prediction MSE."""
    # TODO 5.1: Let B=x0.size(0); draw independent integer t in [0,T)
    # for each example using torch.randint on x0.device.
    # TODO 5.2: Call q_sample(x0,t) to get (x_t,epsilon).
    # TODO 5.3: Predict epsilon_hat=model(x_t,t); compute F.mse_loss.
    # TODO 5.4: zero_grad, backward, step, then return the scalar loss.
    # The SAME method should work for x0 of shape [B,2] or [B,1,28,28].
    raise NotImplementedError("TODO 5.1–5.4: reusable DDPM train step")


In [ ]:
# --- SELF-CHECK 5: DDPM training works on 2D toy points -----------------
net=ToyEpsNet(hidden=16).to(device)
optimizer=torch.optim.Adam(net.parameters(),lr=2e-3)
params_before=[p.detach().clone() for p in net.parameters()]
loss=ddpm_train_step(net,optimizer,X[:16].to(device))
assert loss.ndim==0 and torch.isfinite(loss)
assert any(not torch.equal(a,b) for a,b in zip(params_before,net.parameters()))
print("✓ SELF-CHECK 5: 2D DDPM updates parameters")


In [ ]:
# --- Train toy DDPM after SELF-CHECK 5 -----------------------------------
model=ToyEpsNet().to(device)
# TODO 5.5: Instantiate optimizer Adam(lr=2e-3).
raise NotImplementedError("TODO 5.5: toy optimizer")
X_dev=X.to(device)
for step in range(TOY_TRAIN_STEPS):
    # TODO 5.6: Draw a random batch x0 [256,2] from X_dev.
    # Call your ddpm_train_step(model,optimizer,x0); store result as loss.
    raise NotImplementedError("TODO 5.6: toy mini-batch training")
    if step%500==0:
        print(f"step {step} | loss {loss.item():.4f}")


In [ ]:
# --- TODO 6: DDPM ancestral sampler -------------------------------------
@torch.no_grad()
def ancestral_sample(model, shape, T, betas, alphas, alpha_bars, record_every=None):
    """Generate samples from an epsilon network; return (samples,trajectory)."""
    # TODO 6.1: Start x_T ~ N(0,I) of requested `shape` on `device`.
    # If record_every is specified, initialize traj=[initial.cpu().clone()].
    raise NotImplementedError("TODO 6.1: Gaussian prior and trajectory")

    for i in reversed(range(T)):
        # TODO 6.2: Create tensor t=[i]*batch_size as torch.long;
        # predict eps_hat=model(x,t), read alpha_i and alpha_bar_i.
        raise NotImplementedError("TODO 6.2: reverse timestep and noise prediction")
        # TODO 6.3: Implement DDPM reverse mean (see Eq. in markdown):
        # (1/sqrt(alpha_i)) * [x_i - (1-alpha_i)/sqrt(1-alpha_bar_i) * eps_hat].
        raise NotImplementedError("TODO 6.3: posterior mean parameterization")
        # TODO 6.4: If i>0, draw fresh N(0,I) noise and add sqrt(beta_i)*z.
        # If i==0, take the mean with NO extra noise.
        raise NotImplementedError("TODO 6.4: stochastic reverse step")
        if record_every and i % record_every == 0:
            traj.append(x.cpu().clone())
    return x.cpu(), traj

samples, traj = ancestral_sample(model,(1000,2),T,betas,alphas,alpha_bars,
                                 record_every=200)
plt.scatter(samples[:,0],samples[:,1],s=3,alpha=0.6)
plt.title("DDPM samples (2D toy)")
plt.show()


In [ ]:
# --- SELF-CHECK 6: sampler API with deterministic dummy network ----------
class ZeroEps(nn.Module):
    def forward(self,x,t): return torch.zeros_like(x)
s,history=ancestral_sample(ZeroEps(),(32,2),T=8,
    betas=betas[:8],alphas=alphas[:8],alpha_bars=alpha_bars[:8],record_every=2)
assert s.shape==(32,2) and torch.isfinite(s).all()
assert isinstance(history,list) and len(history)==5
assert all(snap.device.type=='cpu' for snap in history)
print("✓ SELF-CHECK 6: ancestral sample and trajectory snapshots")


In [ ]:
# `traj` holds snapshots of the particle cloud at evenly spaced timesteps,
# from pure noise (leftmost) down to the final samples (rightmost).
# Watching the shape emerge is the fastest way to sanity-check the sampler.
fig, axes = plt.subplots(1, len(traj), figsize=(3 * len(traj), 3))
for ax, snap in zip(axes, traj):
    ax.scatter(snap[:, 0], snap[:, 1], s=2)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Noise → data over reverse diffusion")
plt.show()
# OPTIONAL TODO: plot marginal std at each stored trajectory frame.


### 2.2.2 DDPM on MNIST

2D data only needs coordinates + time. Images need spatial inductive bias.
We use a compact conv net with time-conditioned residual blocks instead of a
full U-Net — same mechanism ($\varepsilon$-prediction, identical training
loop), less boilerplate. Swap in a real U-Net later if you want sharper samples.

In [ ]:
# Rescale data to [-1, 1]
transform_img = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2 - 1),
])
mnist_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform_img)
mnist_loader = DataLoader(mnist_ds, batch_size=128, shuffle=True)
# OPTIONAL TODO: confirm min/max of a batch after [-1,1] rescaling.


In [ ]:
# --- TODO 7: U-Net building blocks and skip connections (MNIST) ---------
# The 28->14->7->14->28 layout and module declarations are scaffolded.
# You implement the differentiable data path through these components.

class ResBlock(nn.Module):
    """Time-conditioned residual block (GroupNorm, convs, time bias, skip)."""
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, temb):
        # TODO 7.1: norm1 -> SiLU -> conv1.
        # Add projected temb [B,out_ch,1,1], then norm2 -> SiLU -> conv2.
        # Return result + skip(x). Keep spatial resolution unchanged.
        raise NotImplementedError("TODO 7.1: ResBlock.forward")


class SelfAttention(nn.Module):
    """Single-head scaled dot-product self-attention on 2D feature maps."""
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.qkv = nn.Conv2d(ch, ch*3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        # TODO 7.2: Let B,C,H,W=x.shape, normalize and split q,k,v.
        # Flatten H*W into sequence length: q,v [B,HW,C], k [B,C,HW].
        # attn = softmax(q@k / sqrt(C), dim=-1) of shape [B,HW,HW].
        # result = attn@v, reshape back to [B,C,H,W]; project and add x.
        # NOTE: tensor.reshape and tensor.permute are useful here.
        raise NotImplementedError("TODO 7.2: image self attention")


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.op = nn.Conv2d(ch,ch,3,stride=2,padding=1)
    def forward(self,x):
        # TODO 7.3: Apply the learned strided convolution.
        raise NotImplementedError("TODO 7.3: downsample")


class Upsample(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2,mode="nearest")
        self.conv=nn.Conv2d(ch,ch,3,padding=1)
    def forward(self,x):
        # TODO 7.4: Upsample by nearest neighbor, then smooth with conv.
        raise NotImplementedError("TODO 7.4: upsample")


class UNet(nn.Module):
    """MNIST epsilon predictor with 28/14/7 resolution, attention at 7x7."""
    def __init__(self,base_ch=64,t_dim=256):
        super().__init__()
        self.temb=nn.Sequential(
            SinusoidalEmbed(t_dim),nn.Linear(t_dim,t_dim),nn.SiLU(),nn.Linear(t_dim,t_dim)
        )
        ch1,ch2,ch3=base_ch,base_ch*2,base_ch*4
        self.in_conv=nn.Conv2d(1,ch1,3,padding=1)
        self.down1_res=ResBlock(ch1,ch1,t_dim)
        self.down1_ds=Downsample(ch1)
        self.down2_res=ResBlock(ch1,ch2,t_dim)
        self.down2_ds=Downsample(ch2)
        self.down3_res=ResBlock(ch2,ch3,t_dim)
        self.mid_res1=ResBlock(ch3,ch3,t_dim)
        self.mid_attn=SelfAttention(ch3)
        self.mid_res2=ResBlock(ch3,ch3,t_dim)
        self.up3_res=ResBlock(ch3+ch3,ch2,t_dim)
        self.up3_us=Upsample(ch2)
        self.up2_res=ResBlock(ch2+ch2,ch1,t_dim)
        self.up2_us=Upsample(ch1)
        self.up1_res=ResBlock(ch1+ch1,ch1,t_dim)
        self.out_norm=nn.GroupNorm(8,ch1)
        self.out_conv=nn.Conv2d(ch1,1,3,padding=1)

    def forward(self,x,t):
        # TODO 7.5: Make temb=self.temb(t).
        # Encoder: h=in_conv(x); h1=down1_res(h,temb) [28]; downsample;
        # h2=down2_res(...,temb) [14]; downsample;
        # h3=down3_res(...,temb) [7]. Preserve h1,h2,h3 as skips.
        # Bottleneck: mid_res1 -> mid_attn -> mid_res2 at 7x7.
        # Decoder: concat h3 to h, apply up3_res -> up3_us (14);
        # concat h2, up2_res -> up2_us (28);
        # concat h1, up1_res. Concatenate along CHANNEL dim=1.
        # Finish: out_conv(SiLU(out_norm(h))) -> predicted noise [B,1,28,28].
        # Check channel widths before each concatenation.
        raise NotImplementedError("TODO 7.5: U-Net forward with skip connections")


In [ ]:
# --- SELF-CHECK 7: U-Net feature sizes and gradient flow -----------------
# Uses a tiny U-Net, so this test works on CPU and without downloading MNIST.
x=torch.randn(2,1,28,28,device=device)
t=torch.tensor([8,900],device=device)
emb=torch.randn(2,32,device=device)
rb=ResBlock(8,16,32).to(device)
assert rb(torch.randn(2,8,14,14,device=device),emb).shape==(2,16,14,14)
attn=SelfAttention(16).to(device)
assert attn(torch.randn(2,16,7,7,device=device)).shape==(2,16,7,7)
assert Downsample(8).to(device)(torch.randn(2,8,28,28,device=device)).shape==(2,8,14,14)
assert Upsample(8).to(device)(torch.randn(2,8,14,14,device=device)).shape==(2,8,28,28)
net=UNet(base_ch=8,t_dim=32).to(device)
output=net(x,t)
assert output.shape==x.shape and torch.isfinite(output).all()
output.square().mean().backward()
assert any(p.grad is not None for p in net.parameters())
print("✓ SELF-CHECK 7: U-Net shape consistency and gradients")


In [ ]:
# --- TODO 8: reuse your DDPM trainer on MNIST ---------------------------
# Do NOT duplicate the DDPM loss: it is the same for 2D and images.
# The shared q_sample and ddpm_train_step handle broadcasting for you.
# First run SELF-CHECK 8 with a small U-Net to debug on CPU.


In [ ]:
# --- SELF-CHECK 8: DDPM trainer works on images too ---------------------
net=UNet(base_ch=8,t_dim=32).to(device)
optimizer=torch.optim.Adam(net.parameters(),lr=2e-4)
params_before=[p.detach().clone() for p in net.parameters()]
small_images=torch.randn(2,1,28,28,device=device)
loss=ddpm_train_step(net,optimizer,small_images)
assert loss.ndim==0 and torch.isfinite(loss)
assert any(not torch.equal(a,b) for a,b in zip(params_before,net.parameters()))
print("✓ SELF-CHECK 8: shared epsilon trainer updates a U-Net")


In [ ]:
# --- Train image DDPM after SELF-CHECK 8 ---------------------------------
img_model=UNet(base_ch=64).to(device)
# TODO 8.1: Create Adam optimizer with lr=2e-4.
raise NotImplementedError("TODO 8.1: image optimizer")
for epoch in range(MNIST_EPOCHS):
    for x0,_ in mnist_loader:
        x0=x0.to(device)
        # TODO 8.2: Use SAME ddpm_train_step(img_model,optimizer,x0).
        raise NotImplementedError("TODO 8.2: train image DDPM")
    print(f"epoch {epoch} | loss {loss.item():.4f}")


In [ ]:
# --- TODO 9: image DDPM ancestral sampling -------------------------------
@torch.no_grad()
def ancestral_sample_img(model, n, T, betas, alphas, alpha_bars):
    """Image version of the SAME reverse kernel used in the toy experiment."""
    # TODO 9.1: Start x from standard normal [n,1,28,28] on `device`.
    raise NotImplementedError("TODO 9.1: image Gaussian prior")
    for i in reversed(range(T)):
        # TODO 9.2: Make a batch of timestep i, evaluate eps_hat, and
        # calculate reverse mean using alpha_i and alpha_bar_i.
        # TODO 9.3: Add sqrt(beta_i)*N(0,I) if i>0, otherwise no extra noise.
        # The formula is exactly the same as ancestral_sample for 2D;
        # change tensor shape only, NOT probability math.
        raise NotImplementedError("TODO 9.2–9.3: image reverse transition")
    return x.cpu()

imgs=ancestral_sample_img(img_model,16,T,betas,alphas,alpha_bars)
imgs=(imgs.clamp(-1,1)+1)/2
fig,axes=plt.subplots(2,8,figsize=(10,3))
for i,ax in enumerate(axes.flat):
    ax.imshow(imgs[i,0],cmap="gray"); ax.axis("off")
plt.suptitle("DDPM samples (MNIST)")
plt.show()


In [ ]:
# --- SELF-CHECK 9: image sampling shapes with zero-noise predictor -------
class ZeroImageEps(nn.Module):
    def forward(self,x,t): return torch.zeros_like(x)
s=ancestral_sample_img(ZeroImageEps(),4,8,betas[:8],alphas[:8],alpha_bars[:8])
assert s.shape==(4,1,28,28) and s.device.type=='cpu'
assert torch.isfinite(s).all()
print("✓ SELF-CHECK 9: image sampler shape and finite output")
